# 03 — Network Prediction Model

**Goal:** Train and compare ML models to predict network behaviour in a refugee camp — bandwidth usage, latency spikes, congestion — so the Pi can pre-allocate resources.

**Input:** `../data/network_logs/` — CSV logs collected by the Raspberry Pi (or synthetic data if not yet collected)

**Output:** `../models/network_model.pkl` — best trained model ready for Pi deployment

## 1. Imports & Config

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

NETWORK_LOGS = Path('../data/network_logs')
MODELS_DIR   = Path('../models')
MODELS_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Imports OK")

## 2. Load or Generate Network Data

If your Pi hasn't started logging yet, synthetic data is generated so you can build and validate the full pipeline now.

In [ ]:
log_files = list(NETWORK_LOGS.glob('*.csv'))

if log_files:
    df = pd.concat([pd.read_csv(f) for f in log_files], ignore_index=True)
    DATA_SOURCE = 'pi_logs'
    print(f"Loaded {len(log_files)} log file(s): {df.shape}")
else:
    print("No Pi logs found — generating synthetic network data")
    DATA_SOURCE = 'synthetic'

    n = 5000
    timestamps = pd.date_range('2024-01-01', periods=n, freq='5min')

    # Simulate realistic camp network patterns
    hour       = timestamps.hour
    day_of_week = timestamps.dayofweek
    # Peak usage: 8am-12pm and 6pm-10pm
    peak_signal = np.sin(2 * np.pi * hour / 24) * 0.5 + 0.5

    df = pd.DataFrame({
        'timestamp':           timestamps,
        'hour':                hour,
        'day_of_week':         day_of_week,
        'active_devices':      (50 + 200 * peak_signal + np.random.normal(0, 20, n)).clip(0).astype(int),
        'bandwidth_mbps':      (2 + 48 * peak_signal + np.random.normal(0, 5, n)).clip(0),
        'latency_ms':          (10 + 200 * (1 - peak_signal) + np.random.exponential(15, n)).clip(5),
        'packet_loss_pct':     (0.5 * (1 - peak_signal) + np.random.exponential(0.3, n)).clip(0, 10),
        'signal_strength_dbm': (-60 - 30 * np.random.rand(n) + np.random.normal(0, 3, n)),
        'connected_users':     (20 + 80 * peak_signal + np.random.normal(0, 10, n)).clip(0).astype(int),
        'temperature_c':       (28 + 10 * np.sin(2 * np.pi * hour / 24) + np.random.normal(0, 2, n)),
    })

    # Target: bandwidth usage 30 minutes ahead
    df['bandwidth_next_30m'] = df['bandwidth_mbps'].shift(-6).fillna(method='ffill')

    print(f"Synthetic network data: {df.shape}")

display(df.head())

## 3. Explore the Network Data

In [ ]:
print(df.describe().T.to_string())

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns.tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
plot_cols = ['bandwidth_mbps', 'latency_ms', 'packet_loss_pct',
             'active_devices', 'connected_users', 'signal_strength_dbm']
plot_cols = [c for c in plot_cols if c in df.columns]

for ax, col in zip(axes.flat, plot_cols):
    df[col].hist(bins=40, ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(col)
    ax.set_xlabel('')

plt.suptitle('Network Feature Distributions', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Bandwidth usage by hour of day
if 'hour' in df.columns and 'bandwidth_mbps' in df.columns:
    hourly = df.groupby('hour')['bandwidth_mbps'].mean()

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(hourly.index, hourly.values, color='cornflowerblue', edgecolor='white')
    ax.set_title('Average Bandwidth Usage by Hour', fontsize=13)
    ax.set_xlabel('Hour of Day')
    ax.set_ylabel('Avg Bandwidth (Mbps)')
    ax.set_xticks(range(24))
    plt.tight_layout()
    plt.show()

## 4. Feature Engineering

In [ ]:
# ── Define target and feature columns — update if your CSV has different names ─
TARGET_COL = 'bandwidth_next_30m'   # what we predict: bandwidth 30 min ahead
                                     # change to 'latency_ms' if predicting latency

if TARGET_COL not in df.columns:
    # Fallback: predict current bandwidth if future target doesn't exist
    TARGET_COL = 'bandwidth_mbps'
    print(f"Target column not found — falling back to '{TARGET_COL}'")

# ─────────────────────────────────────────────────────────────────────────────

df_feat = df.copy()

# Time-based cyclical encoding
if 'hour' in df_feat.columns:
    df_feat['hour_sin'] = np.sin(2 * np.pi * df_feat['hour'] / 24)
    df_feat['hour_cos'] = np.cos(2 * np.pi * df_feat['hour'] / 24)

if 'day_of_week' in df_feat.columns:
    df_feat['dow_sin'] = np.sin(2 * np.pi * df_feat['day_of_week'] / 7)
    df_feat['dow_cos'] = np.cos(2 * np.pi * df_feat['day_of_week'] / 7)

# Rolling features (if enough rows)
if 'bandwidth_mbps' in df_feat.columns and len(df_feat) > 12:
    df_feat['bandwidth_roll_mean_1h'] = df_feat['bandwidth_mbps'].rolling(12, min_periods=1).mean()
    df_feat['bandwidth_roll_std_1h']  = df_feat['bandwidth_mbps'].rolling(12, min_periods=1).std().fillna(0)

# Drop non-numeric / timestamp columns
drop_cols = ['timestamp'] + [TARGET_COL]
FEATURE_COLS = [
    c for c in df_feat.select_dtypes(include='number').columns
    if c not in drop_cols
]

print(f"Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"Target: {TARGET_COL}")

## 5. Train/Test Split

In [ ]:
X = df_feat[FEATURE_COLS].fillna(0)
y = df_feat[TARGET_COL].fillna(0)

# Time-series aware split — keep temporal order
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

## 6. Train & Compare Models

In [ ]:
MODELS = {
    'Ridge':             Ridge(),
    'RandomForest':      RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE),
    'GradientBoosting':  GradientBoostingRegressor(n_estimators=100, random_state=RANDOM_STATE),
    'XGBoost':           XGBRegressor(n_estimators=100, random_state=RANDOM_STATE, verbosity=0),
    'LightGBM':          LGBMRegressor(n_estimators=100, random_state=RANDOM_STATE, verbose=-1),
}

results = []

for name, model in MODELS.items():
    use_scaled = name == 'Ridge'
    Xtr = X_train_sc if use_scaled else X_train
    Xte = X_test_sc  if use_scaled else X_test

    model.fit(Xtr, y_train)
    preds = model.predict(Xte)

    mae  = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2   = r2_score(y_test, preds)

    results.append({'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2})
    print(f"{name:<20}  MAE={mae:.3f}  RMSE={rmse:.3f}  R²={r2:.4f}")

results_df = pd.DataFrame(results).sort_values('R2', ascending=False)
display(results_df)

## 7. Visualise Model Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, metric in zip(axes, ['R2', 'MAE', 'RMSE']):
    ascending = metric != 'R2'
    data = results_df.sort_values(metric, ascending=ascending)
    colors = ['#e74c3c' if i == 0 else '#e67e22' for i in range(len(data))]
    ax.barh(data['Model'], data[metric], color=colors)
    ax.set_title(metric)
    ax.set_xlabel(metric)

plt.suptitle('Network Prediction — Model Comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 8. Prediction Timeline Plot

In [ ]:
best_name  = results_df.iloc[0]['Model']
best_model = MODELS[best_name]
use_scaled = best_name == 'Ridge'
preds_best = best_model.predict(X_test_sc if use_scaled else X_test)

# Show last 200 test points
n_show = min(200, len(y_test))
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(range(n_show), y_test.values[-n_show:],    label='Actual',    linewidth=1.5)
ax.plot(range(n_show), preds_best[-n_show:], label='Predicted', linewidth=1.5, linestyle='--', alpha=0.85)
ax.set_title(f'Network Prediction — Actual vs Predicted ({best_name})')
ax.set_xlabel('Time steps')
ax.set_ylabel(TARGET_COL)
ax.legend()
plt.tight_layout()
plt.show()

## 9. Feature Importance

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importance = pd.Series(best_model.feature_importances_, index=FEATURE_COLS)
    importance = importance.sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(10, 6))
    importance.plot(kind='barh', ax=ax, color='tomato')
    ax.set_title(f'Feature Importance — {best_name}', fontsize=13)
    ax.set_xlabel('Importance')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("Model does not expose feature importances.")

## 10. Save Model & Scaler

In [ ]:
joblib.dump(best_model, MODELS_DIR / 'network_model.pkl')
joblib.dump(scaler,     MODELS_DIR / 'network_scaler.pkl')

meta = {
    'model_type':   best_name,
    'features':     FEATURE_COLS,
    'target':       TARGET_COL,
    'r2_score':     float(results_df.iloc[0]['R2']),
    'mae':          float(results_df.iloc[0]['MAE']),
    'data_source':  DATA_SOURCE,
}
(MODELS_DIR / 'network_model_meta.json').write_text(json.dumps(meta, indent=2))

print("Saved:")
print("  models/network_model.pkl")
print("  models/network_scaler.pkl")
print("  models/network_model_meta.json")
print("\nBoth models trained. Next: populate src/ with production code for Pi.")